In [2]:
import json
import random

import pandas as pd
from datasets import load_dataset

DATASET = "KrishPS/codenet-accepted-c"
SPLIT = "train"
MUST_INCLUDE = "p00000"
N_PROBLEMS = 100
CSV_PATH = "data.csv"
RANDOM_SEED = 42  # change for a different random draw

random.seed(RANDOM_SEED)

print(f"Loading {DATASET} …")
ds = load_dataset(DATASET, split=SPLIT)
df = ds.to_pandas()

# One row per problem (first submission in dataset order; tests are identical per problem).
by_prob = df.drop_duplicates(subset=["problem_id"], keep="first").reset_index(drop=True)

fixed = by_prob[by_prob["problem_id"] == MUST_INCLUDE]
if fixed.empty:
    raise ValueError(f"{MUST_INCLUDE!r} not found in dataset — check problem_id spelling.")

pool = by_prob[by_prob["problem_id"] != MUST_INCLUDE]
k = N_PROBLEMS - 1
if len(pool) < k:
    raise ValueError(f"Need {k} other problems but pool has only {len(pool)}")

sample_idx = pool.sample(n=k, random_state=RANDOM_SEED).index.tolist()
out = pd.concat([fixed, pool.loc[sample_idx]], ignore_index=True)

export = pd.DataFrame(
    {
        "problem_id": out["problem_id"],
        "answer": out["code"],
        "test_cases": out["test_cases"],
    }
)

export['lines-of-code'] = export['answer'].apply(lambda x: len(x.split('\n')))

export.to_csv(CSV_PATH, index=False, encoding="utf-8")
print(f"Wrote {len(export)} rows → {CSV_PATH}")

Loading KrishPS/codenet-accepted-c …
Wrote 100 rows → data.csv
